# LiteLLM、Milvus 与 Langfuse 使用说明

本文档包含三部分内容：

1. LiteLLM 的核心优点与 OpenAI 标准 SDK 调用方式
2. Milvus 的核心功能、建库建表、录入数据与检索方式示例
3. Langfuse 的作用、Prompt 管理与和 LiteLLM 结合的调用方式


## LiteLLM 的优点

LiteLLM 可以作为统一的大模型网关，对外提供 **OpenAI 兼容接口**，在工程落地中有几个明显优势：

- **统一接口，迁移成本低**：业务侧继续使用 OpenAI 标准 SDK，只需要把 `base_url` 指向 LiteLLM，即可接入不同厂商模型。
- **多模型统一接入**：可以在一个入口下管理 OpenAI、DeepSeek、Qwen、Claude 等不同模型，避免每家厂商单独维护一套接入代码。
- **便于切换底层模型**：业务代码不需要跟具体厂商强绑定，只要 LiteLLM 里配置了模型映射，就可以灵活替换。
- **支持路由与容灾**：可以配置 fallback、负载均衡、限流和重试，提升整体可用性。
- **成本与权限更容易管理**：可以统一做 API Key 管理、预算控制、调用审计和使用统计。
- **便于企业内部标准化**：把模型访问统一收口到网关层，业务项目只依赖一套调用方式，后续维护更简单。

http://localhost:4000/

## OpenAI 标准 SDK 调用 LiteLLM

下面示例继续使用 OpenAI 官方 Python SDK，但把 `base_url` 和 `api_key` 改成 LiteLLM 的地址与密钥。

> 说明：如果是本地 LiteLLM 且未开启严格鉴权，`api_key` 可以填写任意非空字符串；真正生效的是请求发往 `http://localhost:4000/v1`。


In [16]:
import os

from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.getenv("LITELLM_API_KEY", "sk-local-litellm"),
    base_url=os.getenv("LITELLM_BASE_URL", "http://localhost:4000/v1"),
)

response = client.chat.completions.create(
    model="deepseek-chat",  # 替换为 LiteLLM 中实际配置的模型名
    messages=[
        {"role": "system", "content": "你是一个简洁的助手。"},
        {"role": "user", "content": "请用一句话介绍 LiteLLM 的作用。"},
    ],
    temperature=0.3,
)

print(response.choices[0].message.content)


LiteLLM 是一个统一接口库，可让开发者用相同方式调用多种大语言模型服务。


In [14]:
import os

from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("LITELLM_API_KEY", "sk-local-litellm"),
    base_url=os.getenv("LITELLM_BASE_URL", "http://localhost:4000/v1"),
)

response = client.chat.completions.create(
    model="qwen-plus",  # 替换为 LiteLLM 中实际配置的模型名
    messages=[
        {"role": "system", "content": "你是一个简洁的助手。"},
        {"role": "user", "content": "请用一句话介绍 LiteLLM 的作用。"},
    ],
    temperature=0.3,
)

print(response.choices[0].message.content)


LiteLLM 是一个轻量级的开源库，用于统一调用多种大语言模型（如 OpenAI、Anthropic、Gemini、本地 LLM 等）的 API，提供标准化接口、自动模型路由、负载均衡和成本监控等功能。


In [2]:
import os

from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("LITELLM_API_KEY", "sk-local-litellm"),
    base_url=os.getenv("LITELLM_BASE_URL", "http://localhost:4000/v1"),
)

response = client.chat.completions.create(
    model="glm-4-plus",  # 替换为 LiteLLM 中实际配置的模型名
    messages=[
        {"role": "system", "content": "你是一个简洁的助手。"},
        {"role": "user", "content": "请用一句话介绍 LiteLLM 的作用。"},
    ],
    temperature=0.3,
)

print(response.choices[0].message.content)


LiteLLM 是一个统一接口，可让开发者轻松与多种大型语言模型 API 进行交互。


## LiteLLM 补充说明

- 如果你原来就是使用 OpenAI SDK，那么接入 LiteLLM 时通常只需要修改 `base_url` 和 `api_key`。
- `model` 参数填写的是 **LiteLLM 暴露出来的模型名或别名**，不一定必须是底层厂商的原始模型名。
- 如果后续要切换到底层其他模型，优先在 LiteLLM 配置层调整，业务代码尽量保持不变。


## Milvus 的功能

Milvus 是面向向量检索场景的数据库，适合保存题目 embedding、文本内容和业务元数据，并提供高性能相似度搜索能力。

在本项目中，Milvus 主要承担以下职责：

- **保存向量数据**：存储题目文本对应的 embedding，作为语义检索基础。
- **保存元数据**：例如 `paper_id`、`box_id`、`subject_key`，便于过滤和结果回溯。
- **支持相似检索**：根据查询向量找出语义最接近的题目。
- **支持过滤检索**：在向量检索时叠加元数据过滤，例如只查某个学科。
- **支持标量查询**：不做向量召回时，也可以直接按元数据查询具体记录。
- **支持增量更新**：通过 `insert` 或 `upsert` 逐步维护题库数据。

http://localhost:8000/

## Milvus 中 database 与 collection 的关系

- **database**：逻辑命名空间，用来隔离不同业务的数据。
- **collection**：真正存储数据的对象，类似关系型数据库中的表。
- **metadata**：通常作为 collection 里的标量字段保存，而不是单独建一张元数据表。

下面示例会创建一个 `question_match` 数据库，并在其中创建 `question_bank` collection，字段包括：

- 主键：`question_id`
- 原始文本：`question_text`
- 元数据：`paper_id`、`box_id`、`subject_key`
- 向量字段：`embedding`


## 创建 Milvus 数据库与 Collection（包含元数据）

这个示例使用 `MilvusClient` 创建逻辑数据库、collection schema 和索引。向量字段使用 `COSINE` 相似度，元数据字段建立 `INVERTED` 标量索引，便于后续过滤检索。


In [3]:
from pymilvus import DataType, MilvusClient

MILVUS_URI = "http://localhost:19530"
DB_NAME = "question_match"
COLLECTION_NAME = "question_bank"
EMBED_DIM = 1024  # 本项目可与 BGE-M3 保持一致

client = MilvusClient(uri=MILVUS_URI)

if DB_NAME not in client.list_databases():
    client.create_database(db_name=DB_NAME)

client.use_database(db_name=DB_NAME)

if client.has_collection(collection_name=COLLECTION_NAME):
    client.drop_collection(collection_name=COLLECTION_NAME)

schema = MilvusClient.create_schema(
    auto_id=False,
    enable_dynamic_field=False,
)

schema.add_field(
    field_name="question_id",
    datatype=DataType.VARCHAR,
    is_primary=True,
    max_length=64,
)
schema.add_field(
    field_name="question_text",
    datatype=DataType.VARCHAR,
    max_length=8192,
)
schema.add_field(
    field_name="paper_id",
    datatype=DataType.VARCHAR,
    max_length=64,
)
schema.add_field(
    field_name="box_id",
    datatype=DataType.VARCHAR,
    max_length=64,
)
schema.add_field(
    field_name="subject_key",
    datatype=DataType.VARCHAR,
    max_length=32,
)
schema.add_field(
    field_name="embedding",
    datatype=DataType.FLOAT_VECTOR,
    dim=EMBED_DIM,
)

index_params = client.prepare_index_params()
index_params.add_index(
    field_name="embedding",
    index_name="embedding_index",
    index_type="AUTOINDEX",
    metric_type="COSINE",
)
index_params.add_index(
    field_name="paper_id",
    index_name="paper_id_index",
    index_type="INVERTED",
)
index_params.add_index(
    field_name="box_id",
    index_name="box_id_index",
    index_type="INVERTED",
)
index_params.add_index(
    field_name="subject_key",
    index_name="subject_key_index",
    index_type="INVERTED",
)

client.create_collection(
    collection_name=COLLECTION_NAME,
    schema=schema,
    index_params=index_params,
)

print(f"数据库 {DB_NAME} 与集合 {COLLECTION_NAME} 创建完成")


数据库 question_match 与集合 question_bank 创建完成


## 录入数据（包含元数据）

录入时，每条数据通常包含 4 部分：

- 主键 ID
- 原始题目文本
- 业务元数据
- 文本对应的 embedding 向量

下面示例沿用上面的 LiteLLM 网关，使用 OpenAI 标准 SDK 获取 embedding，然后写入 Milvus。

> 说明：`embedding_model` 需要替换为你在 LiteLLM 中实际暴露的 embedding 模型名。


In [4]:
import os

from openai import OpenAI

embed_client = OpenAI(
    api_key=os.getenv("LITELLM_API_KEY", "sk-local-litellm"),
    base_url=os.getenv("LITELLM_BASE_URL", "http://localhost:4000/v1"),
)

EMBEDDING_MODEL = "bge-m3"  # 替换为 LiteLLM 中配置的 embedding 模型别名


def embed_text(text: str) -> list[float]:
    response = embed_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text,
    )
    return response.data[0].embedding


rows = [
    {
        "question_id": "q_001",
        "question_text": "已知二次函数 y=ax^2+bx+c，求其顶点坐标。",
        "paper_id": "paper_1001",
        "box_id": "box_01",
        "subject_key": "数学",
    },
    {
        "question_id": "q_002",
        "question_text": "求抛物线的顶点式，并说明顶点坐标如何计算。",
        "paper_id": "paper_1001",
        "box_id": "box_02",
        "subject_key": "数学",
    },
    {
        "question_id": "q_003",
        "question_text": "《岳阳楼记》的作者是谁？请写出朝代。",
        "paper_id": "paper_1002",
        "box_id": "box_03",
        "subject_key": "语文",
    },
]

entities = []
for row in rows:
    entity = row | {"embedding": embed_text(row["question_text"])}
    entities.append(entity)

insert_result = client.insert(collection_name=COLLECTION_NAME, data=entities)
print(insert_result)


{'insert_count': 30, 'ids': ['q_001', 'q_002', 'q_003', 'q_004', 'q_005', 'q_006', 'q_007', 'q_008', 'q_009', 'q_010', 'q_011', 'q_012', 'q_013', 'q_014', 'q_015', 'q_016', 'q_017', 'q_018', 'q_019', 'q_020', 'q_021', 'q_022', 'q_023', 'q_024', 'q_025', 'q_026', 'q_027', 'q_028', 'q_029', 'q_030']}


### 增量更新示例

如果题目已经存在，但文本或元数据发生变化，可以使用 `upsert` 做幂等更新。


In [5]:
updated_row = {
    "question_id": "q_002",
    "question_text": "求二次函数的顶点式，并写出顶点坐标的求法。",
    "paper_id": "paper_1001",
    "box_id": "box_02",
    "subject_key": "数学",
}
updated_row["embedding"] = embed_text(updated_row["question_text"])

upsert_result = client.upsert(
    collection_name=COLLECTION_NAME,
    data=[updated_row],
)

print(upsert_result)


{'upsert_count': 1, 'primary_keys': ['q_002']}


## 检索数据方式

Milvus 常见检索方式可以分为三类：

- **向量相似检索**：只根据 embedding 找最相近结果。
- **向量检索 + 元数据过滤**：语义召回时限制学科、试卷、题块等范围。
- **标量查询**：只按元数据字段直接查，不经过向量相似度计算。


### 1. 向量相似检索

适合输入一句新题目，直接查找语义上最相近的历史题目。


In [15]:
query_text = "岳阳楼记 数学公式 "
query_vector = embed_text(query_text)

search_result = client.search(
    collection_name=COLLECTION_NAME,
    data=[query_vector],
    anns_field="embedding",
    limit=3,
    output_fields=["question_text", "paper_id", "box_id", "subject_key"],
    search_params={"metric_type": "COSINE", "params": {}},
)

for item in search_result[0]:
    print(item)


AttributeError: 'OpenAI' object has no attribute 'search'

### 2. 向量检索 + 元数据过滤

适合在某个业务范围内检索，例如“只在数学题里做语义匹配”。


In [10]:
filtered_result = client.search(
    collection_name=COLLECTION_NAME,
    data=[query_vector],
    anns_field="embedding",
    limit=3,
    filter='subject_key == "数学"',
    output_fields=["question_text", "paper_id", "box_id", "subject_key"],
    search_params={"metric_type": "COSINE", "params": {}},
)

for item in filtered_result[0]:
    print(item)


{'question_id': 'q_002', 'distance': 0.4756309986114502, 'entity': {'question_text': '求二次函数的顶点式，并写出顶点坐标的求法。', 'paper_id': 'paper_1001', 'box_id': 'box_02', 'subject_key': '数学'}}
{'question_id': 'q_001', 'distance': 0.4594928026199341, 'entity': {'question_text': '已知二次函数 y=ax^2+bx+c，求其顶点坐标。', 'paper_id': 'paper_1001', 'box_id': 'box_01', 'subject_key': '数学'}}
{'question_id': 'q_007', 'distance': 0.44382232427597046, 'entity': {'question_text': '计算：(a+b)^2 - (a-b)^2。', 'paper_id': 'paper_1002', 'box_id': 'box_02', 'subject_key': '数学'}}


### 3. 只按元数据做标量查询

适合根据 `paper_id`、`box_id` 等字段直接取数，不做向量相似度召回。


In [9]:
query_result = client.query(
    collection_name=COLLECTION_NAME,
    filter='paper_id == "paper_1001"',
    output_fields=["question_id", "question_text", "box_id", "subject_key"],
)

for item in query_result:
    print(item)


{'box_id': 'box_01', 'subject_key': '数学', 'question_id': 'q_001', 'question_text': '已知二次函数 y=ax^2+bx+c，求其顶点坐标。'}
{'box_id': 'box_02', 'subject_key': '数学', 'question_id': 'q_002', 'question_text': '求二次函数的顶点式，并写出顶点坐标的求法。'}
{'box_id': 'box_03', 'subject_key': '数学', 'question_id': 'q_004', 'question_text': '已知一次函数 y=2x-3，求当 x=4 时的函数值。'}
{'box_id': 'box_04', 'subject_key': '数学', 'question_id': 'q_005', 'question_text': '解方程：2x^2-5x-3=0。'}


## Milvus 使用小结

在这套流程里：

1. 先在 Milvus 中创建 **database + collection schema + index**。
2. 将题目文本转成 embedding，并与元数据一起写入。
3. 查询时优先走 `search` 做向量相似检索；如果需要业务约束，就叠加 `filter`。
4. 如果只是按元数据取数，则直接用 `query`。

这也是试题匹配、RAG 检索和 OCR 后处理场景里最常见的 Milvus 使用模式。


## Langfuse 的作用

Langfuse 不是模型网关，而是 LLM 应用里的 **Prompt 管理 + 可观测性** 平台。

在工程里它主要解决 4 类问题：

- **Prompt 版本管理**：把 system/user prompt 放到平台里管理，支持按 `name`、`version`、`label` 拉取。
- **调用链追踪**：记录一次请求里的输入、输出、耗时、错误、上下文和模型调用链。
- **成本与效果分析**：查看 token、成本、响应质量，便于做版本对比。
- **评测闭环**：后续可以接 dataset、人工标注和评分，持续优化 prompt 与模型。

和本笔记本中其他组件的分工通常是：

- **LiteLLM**：统一模型访问入口，负责把请求路由到底层模型。
- **Langfuse**：管理 prompt、trace、版本和评测。
- **OpenAI SDK**：提供统一调用接口，减少业务代码改造量。

http://localhost:3100/project/cmm06mxw70006oy07ribk4qp4/prompts/get_country_chat

## Langfuse 的常见用法

最常见的接入方式有两类：

1. **Prompt Management**：在 Langfuse 控制台创建 prompt，代码里通过 `get_prompt()` 拉取，再用 `compile()` 填充变量。
2. **Tracing / Observability**：用 `@observe()` 给业务函数加 trace，或使用 `from langfuse.openai import OpenAI` 对 OpenAI SDK 做无侵入埋点。

常用环境变量示例：

```bash
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_BASE_URL=https://cloud.langfuse.com
LITELLM_BASE_URL=http://localhost:4000/v1
LITELLM_API_KEY=sk-local-litellm
```

> 说明：
> - 自建 Langfuse 时把 `LANGFUSE_BASE_URL` 改成你的服务地址。
> - `LANGFUSE_HOST` 仍可兼容旧版本，但新代码优先使用 `LANGFUSE_BASE_URL`。
> - LiteLLM 本地未开启鉴权时，`LITELLM_API_KEY` 可以是任意非空字符串；Langfuse 的 key 仍需要真实值，否则无法拉取 prompt 和写入 trace。


## Langfuse Prompt 管理最小示例

下面示例只演示 Langfuse 的 Prompt Management 能力。它从 Langfuse 读取一个 chat prompt，并用变量 `country` 编译成 OpenAI 兼容的 `messages`。


In [19]:
from langfuse import Langfuse

langfuse = Langfuse()  # 默认从环境变量读取 Langfuse 配置

# 在 Langfuse 控制台中先创建一个名为 get_country_chat 的 chat prompt
prompt = langfuse.get_prompt("get_country_chat", type="chat")
messages = prompt.compile(country="中国")

print("Messages:")
for message in messages:
    print(message)


Messages:
{'role': 'system', 'content': '你是一个地理专家，准确精炼的给用户说首都的名称，不输出其它的文字。'}
{'role': 'user', 'content': '请告诉我 中国 的首都是什么？'}


## Langfuse + LiteLLM 的实际使用样例

下面这个例子把三层串起来：

- **Langfuse** 托管 prompt 模板。
- **OpenAI SDK** 用标准接口发起请求。
- **LiteLLM** 作为统一网关，把请求转发到底层模型。
- `@observe()` 和 `langfuse.openai` 负责把这次调用记录到 Langfuse。

> 建议先在 Langfuse 控制台创建一个 chat prompt `get_country_chat`，例如：
> - `system`: `你是一个简洁的地理助手。`
> - `user`: `请介绍 {{country}} 的首都、人口和一个代表性城市。`


In [20]:
import os

from langfuse import Langfuse, observe
from langfuse.openai import OpenAI

langfuse = Langfuse()  # 默认从环境变量读取 Langfuse 配置

client = OpenAI(
    api_key=os.getenv("LITELLM_API_KEY", "sk-local-litellm"),
    base_url=os.getenv("LITELLM_BASE_URL", "http://localhost:4000/v1"),
)

@observe(name="country-chat-workflow")
def ask_country(country: str) -> str:
    prompt = langfuse.get_prompt("get_country_chat", type="chat")
    messages = prompt.compile(country=country)

    print("Messages:", messages)

    response = client.chat.completions.create(
        model="qwen-plus",  # 替换为 LiteLLM 中实际配置的模型名
        messages=messages,
        temperature=0.3,
    )
    return response.choices[0].message.content or ""

result = ask_country("中国")
print("Output:", result)

# notebook 中同步刷出 trace，便于立即在 Langfuse 控制台查看
langfuse.flush()


Messages: [{'role': 'system', 'content': '你是一个地理专家，准确精炼的给用户说首都的名称，不输出其它的文字。'}, {'role': 'user', 'content': '请告诉我 中国 的首都是什么？'}]
Output: 北京


## Langfuse 使用小结

在这套组合里，一个比较稳妥的实践方式是：

1. 把 prompt 模板放到 Langfuse 控制台做版本管理。
2. 业务代码里只保留 `prompt name + variables`，通过 `compile()` 生成最终消息。
3. 模型调用统一走 LiteLLM，便于底层模型切换和鉴权管理。
4. 用 `@observe()` 或 `langfuse.openai` 自动记录 trace，后续再结合评测与人工标注优化效果。
